# RNFLT features: MLP vs. XGBoost

Compare MLP and XGBoost models using the 9 numeric predictors in `rnflt_features.csv`, with and without age. Race is excluded from model inputs and used only for subgroup metrics. The official train/validation/test splits are preserved (2,100 / 300 / 900); preprocessing is fitted on training data only. The validation split is used for early stopping, and AUROC and 10-bin ECE are reported by race and overall on both validation and test.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

DATA_DIR = next(
    (path for path in (Path("data/processed"), Path("../data/processed")) if path.is_dir()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find data/processed from the notebook working directory.")

meta = pd.read_csv(DATA_DIR / "meta.csv")
rnflt = pd.read_csv(DATA_DIR / "rnflt_features.csv")
if "filename" not in meta.columns or "filename" not in rnflt.columns:
    raise ValueError("Both CSV files must contain filename for row alignment.")
data = meta.merge(rnflt, on="filename", how="inner", validate="one_to_one")
if len(data) != len(meta) or len(data) != len(rnflt):
    raise ValueError("Metadata and RNFLT feature filenames/counts do not match.")

rnflt_columns = [column for column in rnflt.columns if column != "filename"]
if len(rnflt_columns) != 9:
    raise ValueError(f"Expected 9 RNFLT feature columns, found {len(rnflt_columns)}.")
feature_sets = {
    "RNFLT + age": ["age", *rnflt_columns],
    "RNFLT only (no age)": rnflt_columns,
}
for columns in feature_sets.values():
    data[columns] = data[columns].apply(pd.to_numeric, errors="coerce")
data["label"] = data["glaucoma"].astype(str).str.strip().str.lower().map({"no": 0, "yes": 1})
data["race"] = data["race"].astype(str).str.strip().str.lower()
if data["label"].isna().any():
    raise ValueError("glaucoma must contain only 'yes' and 'no'.")
if "split" in data.columns:
    data["split"] = data["split"].astype(str).str.strip().str.lower().replace({"validation": "val"})
elif "use" in data.columns:
    data["split"] = data["use"].astype(str).str.strip().str.lower().map(
        {"training": "train", "validation": "val", "test": "test"}
    )
else:
    raise ValueError("meta.csv must contain split or use.")
if data["split"].isna().any() or set(data["split"].unique()) != {"train", "val", "test"}:
    raise ValueError("Expected official train, val, and test splits.")

print(f"Data: {DATA_DIR.resolve()} | rows by split: {data['split'].value_counts().to_dict()}")
print(f"RNFLT predictors: {rnflt_columns}")
print(f"Missing feature values: {int(data[[*feature_sets['RNFLT + age']]].isna().sum().sum())}")

SEED = 42
BATCH_SIZE = 32
MAX_EPOCHS = 100
PATIENCE = 10
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
train_mask = data["split"].eq("train").to_numpy()
val_mask = data["split"].eq("val").to_numpy()
test_mask = data["split"].eq("test").to_numpy()
y = data["label"].to_numpy(dtype=np.int64)
y_train, y_val, y_test = y[train_mask], y[val_mask], y[test_mask]

class MultiLayerPerceptron(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 2),
        )

    def forward(self, inputs):
        return self.network(inputs)

def train_model(columns):
    raw = data[columns].to_numpy(dtype=np.float32)
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    x_train = scaler.fit_transform(imputer.fit_transform(raw[train_mask])).astype(np.float32)
    x_val = scaler.transform(imputer.transform(raw[val_mask])).astype(np.float32)
    x_test = scaler.transform(imputer.transform(raw[test_mask])).astype(np.float32)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    torch.manual_seed(SEED)
    model = MultiLayerPerceptron(input_size=len(columns)).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    x_val_tensor = torch.from_numpy(x_val).to(DEVICE)
    y_val_tensor = torch.from_numpy(y_val).to(DEVICE)
    best_loss, best_state, best_epoch, stale_epochs = float("inf"), None, 0, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(x_val_tensor), y_val_tensor).item()
        if val_loss < best_loss - 1e-4:
            best_loss = val_loss
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            best_epoch = epoch
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= PATIENCE:
                break
    if best_state is None:
        raise RuntimeError("Training did not produce a valid model checkpoint.")
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_prob = torch.softmax(model(x_val_tensor), dim=1)[:, 1].cpu().numpy()
        test_prob = torch.softmax(model(torch.from_numpy(x_test).to(DEVICE)), dim=1)[:, 1].cpu().numpy()
    return val_prob, test_prob, best_epoch, best_loss

def train_xgboost(columns):
    raw = data[columns].to_numpy(dtype=np.float32)
    imputer = SimpleImputer(strategy="median")
    x_train = imputer.fit_transform(raw[train_mask]).astype(np.float32)
    x_val = imputer.transform(raw[val_mask]).astype(np.float32)
    x_test = imputer.transform(raw[test_mask]).astype(np.float32)
    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.9,
        reg_lambda=2.0,
        early_stopping_rounds=30,
        n_jobs=1,
        random_state=SEED,
        tree_method="hist",
    )
    model.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
    return model.predict_proba(x_val)[:, 1], model.predict_proba(x_test)[:, 1], model.best_iteration

def expected_calibration_error(y_true, probabilities, n_bins=10):
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for index in range(n_bins):
        in_bin = (probabilities >= edges[index]) & (probabilities <= edges[index + 1]) if index == 0 else (
            (probabilities > edges[index]) & (probabilities <= edges[index + 1])
        )
        if in_bin.any():
            correct = np.mean((probabilities[in_bin] >= 0.5) == y_true[in_bin])
            confidence = np.mean(np.maximum(probabilities[in_bin], 1 - probabilities[in_bin]))
            ece += in_bin.mean() * abs(correct - confidence)
    return float(ece)

rows = []
for model_name, columns in feature_sets.items():
    mlp_val_prob, mlp_test_prob, best_epoch, best_loss = train_model(columns)
    xgb_val_prob, xgb_test_prob, best_iteration = train_xgboost(columns)
    print(f"{model_name}: MLP best epoch={best_epoch}; XGBoost best iteration={best_iteration}")
    for algorithm, val_prob, test_prob in (
        ("MLP", mlp_val_prob, mlp_test_prob),
        ("XGBoost", xgb_val_prob, xgb_test_prob),
    ):
        for split_name, split_mask, split_labels, probabilities in (
            ("Validation", val_mask, y_val, val_prob),
            ("Test", test_mask, y_test, test_prob),
        ):
            race_values = data.loc[split_mask, "race"].to_numpy()
            for race in ("white", "black", "asian", "overall"):
                race_mask = np.ones(len(split_labels), dtype=bool) if race == "overall" else race_values == race
                race_labels = split_labels[race_mask]
                race_prob = probabilities[race_mask]
                if np.unique(race_labels).size != 2:
                    raise ValueError(f"{split_name}/{race} subset must contain both label classes.")
                rows.append({
                    "model": f"{algorithm} | {model_name}",
                    "split": split_name,
                    "race": race.title(),
                    "n": int(race_mask.sum()),
                    "glaucoma_cases": int(race_labels.sum()),
                    "AUROC": roc_auc_score(race_labels, race_prob),
                    "ECE_10_bins": expected_calibration_error(race_labels, race_prob),
                })

results_df = pd.DataFrame(rows)
results_df

Data: /Users/freddy/projects/Glaucoma-Classification-Project/data/processed | rows by split: {'train': 2100, 'test': 900, 'val': 300}
RNFLT predictors: ['hcdr', 'rim_px', 'rnfl_mean', 'rnfl_p10', 'ring_mean', 'ring_sup', 'ring_inf', 'ring_horiz', 'si_asym']
Missing feature values: 11
RNFLT + age: MLP best epoch=39; XGBoost best iteration=144
RNFLT only (no age): MLP best epoch=27; XGBoost best iteration=148


                        model      split    race   n  glaucoma_cases    AUROC  ECE_10_bins
            MLP | RNFLT + age Validation   White 100              51 0.834334     0.155764
            MLP | RNFLT + age Validation   Black 100              69 0.808322     0.120394
            MLP | RNFLT + age Validation   Asian 100              56 0.863231     0.097622
            MLP | RNFLT + age Validation Overall 300             176 0.824368     0.046249
            MLP | RNFLT + age       Test   White 300             155 0.882892     0.078663
            MLP | RNFLT + age       Test   Black 300             190 0.793254     0.087014
            MLP | RNFLT + age       Test   Asian 300             144 0.874911     0.066192
            MLP | RNFLT + age       Test Overall 900             489 0.849984     0.032621
        XGBoost | RNFLT + age Validation   White 100              51 0.841937     0.127691
        XGBoost | RNFLT + age Validation   Black 100              69 0.840112     0.090551